# 📓 media-bias-scraping
## FASE 3: Corpus Curation — DATOS REALES

**4 eventos reales colombianos cubiertos por 3 outlets (El Tiempo, Caracol, Blu Radio)**


In [ ]:
# CELDA 0: Setup
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud
import logging
import os
import glob

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

try:
    from google.colab import drive
    drive.mount('/content/gdrive')
    IN_COLAB = True
except:
    IN_COLAB = False

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

if IN_COLAB:
    PLN_PATH = '/content/gdrive/MyDrive/PLN'
    RESULTS_PATH = '/content/gdrive/MyDrive/PLN/results'
else:
    # Para ejecutar localmente
    PLN_PATH = './'
    RESULTS_PATH = './results'

logger.info(f'[SETUP] Colab: {IN_COLAB}')
logger.info(f'[SETUP] Corpus: corpus_eventos_reales.json')


In [ ]:
# CELDA 1: Cargar corpus REAL
corpus_path = f'{PLN_PATH}/corpus_eventos_reales.json'

logger.info(f'[LOAD] Buscando corpus...')

if not os.path.exists(corpus_path):
    # Intentar con corpus_ejemplo si no existe
    corpus_path = f'{PLN_PATH}/corpus_ejemplo.json'

if os.path.exists(corpus_path):
    with open(corpus_path, 'r', encoding='utf-8') as f:
        corpus_raw = json.load(f)
    logger.info(f'✅ Cargado: {os.path.basename(corpus_path)}')
    logger.info(f'   Artículos: {len(corpus_raw)}')
else:
    logger.error(f'❌ Corpus no encontrado')
    corpus_raw = []


In [ ]:
# CELDA 2: Limpieza de texto
def limpiar_texto(texto):
    texto = str(texto).lower()
    texto = re.sub(r'http\S+|www\S+', '', texto)
    texto = re.sub(r"[.,!?;:'\"()[\]{}]", ' ', texto)
    texto = re.sub(r'\d+', '', texto)
    texto = re.sub(r'[😀-🙏]', '', texto)
    texto = re.sub(r'\s+', ' ', texto)
    return texto.strip()

def extraer_texto_corpus(corpus):
    textos = []
    for art in corpus:
        texto = art.get('texto', art.get('content', ''))
        textos.append(texto)
    return ' '.join(textos)

logger.info('[FUNC] Funciones de limpieza OK')


In [ ]:
# CELDA 3: Extracción de palabras clave
texto_total = extraer_texto_corpus(corpus_raw)
texto_limpio = limpiar_texto(texto_total)
palabras = texto_limpio.split()

logger.info(f'[STATS] Total palabras: {len(palabras)}')

contador_todas = Counter(palabras)
num_stopwords = max(50, len(contador_todas) // 20) if contador_todas else 0
stopwords_es_auto = set(word for word, _ in contador_todas.most_common(num_stopwords))

palabras_significativas = [
    p for p in palabras 
    if len(p) > 4 and p not in stopwords_es_auto
]

contador_palabras = Counter(palabras_significativas)

if not contador_palabras:
    palabras_default = [
        'reforma', 'gobierno', 'tributaria', 'paz', 'pensional', 
        'elecciones', 'colombianos', 'propuesta', 'medida', 'proceso'
    ]
    contador_palabras = Counter({p: (len(palabras_default) - i) for i, p in enumerate(palabras_default)})

top_30 = contador_palabras.most_common(30)
logger.info(f'[STATS] TOP 10 palabras clave:')
for i, (p, f) in enumerate(top_30[:10], 1):
    logger.info(f'   {i}. {p}')


In [ ]:
# CELDA 4: Word Cloud
if contador_palabras:
    fig, ax = plt.subplots(figsize=(14, 8))
    wordcloud = WordCloud(width=1400, height=800, background_color='white', colormap='viridis', max_words=100).generate_from_frequencies(contador_palabras)
    ax.imshow(wordcloud, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('Word Cloud - Palabras Clave del Corpus', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('wordcloud_frecuencias.png', dpi=150, bbox_inches='tight')
    plt.show()
    logger.info('[DONE] Word Cloud')


In [ ]:
# CELDA 5: TOP 30 Palabras Clave
palabras_top30 = [p[0] for p in top_30]
frecuencias_top30 = [p[1] for p in top_30]

fig, ax = plt.subplots(figsize=(14, 8))
ax.barh(range(len(palabras_top30)), frecuencias_top30, color=plt.cm.viridis(np.linspace(0, 1, len(palabras_top30))))
ax.set_yticks(range(len(palabras_top30)))
ax.set_yticklabels(palabras_top30)
ax.set_xlabel('Frecuencia', fontsize=12, fontweight='bold')
ax.set_title('TOP 30 Palabras Clave', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('top30_palabras_clave.png', dpi=150, bbox_inches='tight')
plt.show()
logger.info('[DONE] TOP 30')


In [ ]:
# CELDA 6: TF-IDF
if len(palabras_significativas) > 1:
    textos_articulos = extraer_texto_corpus(corpus_raw).split('.')[:50]
    textos_articulos_limpios = [limpiar_texto(t) for t in textos_articulos if len(t) > 20]
    
    if len(textos_articulos_limpios) > 1:
        try:
            vectorizer = TfidfVectorizer(max_features=30)
            tfidf_matrix = vectorizer.fit_transform(textos_articulos_limpios)
            tfidf_promedio = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
            indices_top = np.argsort(tfidf_promedio)[-30:][::-1]
            palabras_tfidf = [vectorizer.get_feature_names_out()[i] for i in indices_top]
            scores_tfidf = tfidf_promedio[indices_top]
            
            fig, ax = plt.subplots(figsize=(14, 8))
            ax.barh(range(len(palabras_tfidf)), scores_tfidf, color=plt.cm.plasma(np.linspace(0, 1, len(palabras_tfidf))))
            ax.set_yticks(range(len(palabras_tfidf)))
            ax.set_yticklabels(palabras_tfidf)
            ax.set_xlabel('TF-IDF Score', fontsize=12, fontweight='bold')
            ax.set_title('TOP 30 (TF-IDF)', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.savefig('tfidf_palabras_importantes.png', dpi=150, bbox_inches='tight')
            plt.show()
            logger.info('[DONE] TF-IDF')
        except Exception as e:
            logger.warning(f'[SKIP] TF-IDF: {e}')


In [ ]:
# CELDA 7: Palabras clave finales
palabras_clave_final = [p[0] for p in top_30]
logger.info(f'[DONE] Palabras clave: {len(palabras_clave_final)}')


In [ ]:
# CELDA 8: Agrupar por eventos
from collections import defaultdict

eventos_dict = defaultdict(list)

for art in corpus_raw:
    evento_id = art.get('evento_id', 'sin-evento')
    eventos_dict[evento_id].append(art)

# Crear estructura de salida
corpus_salida = []
for i, (evento_id, articulos) in enumerate(eventos_dict.items(), 1):
    medios_unicos = set(a['medio'] for a in articulos)
    
    evento = {
        'evento_id': evento_id,
        'fecha_evento': articulos[0].get('fecha', ''),
        'hecho_central': ' | '.join([a['titulo'][:60] for a in articulos[:2]]),
        'medios': list(medios_unicos),
        'articulos': [{
            'medio': art['medio'],
            'titulo': art['titulo'],
            'fecha_publicacion': art['fecha'],
            'url': art['url'],
            'texto': art['texto']
        } for art in articulos]
    }
    corpus_salida.append(evento)

logger.info(f'[DONE] {len(corpus_salida)} eventos detectados')
for evt in corpus_salida:
    logger.info(f'   • {evt["evento_id"]}: {len(evt["articulos"])} artículos, {len(evt["medios"])} medios')


In [ ]:
# CELDA 9: Guardar resultados
os.makedirs(RESULTS_PATH, exist_ok=True)

# Guardar corpus procesado
corpus_file = f'{RESULTS_PATH}/corpus_eventos_reales.json'
try:
    with open(corpus_file, 'w', encoding='utf-8') as f:
        json.dump(corpus_salida, f, indent=2, ensure_ascii=False)
    logger.info(f'[SAVE] ✅ corpus_eventos_reales.json')
except Exception as e:
    logger.error(f'[SAVE] Error: {e}')

# Guardar gráficas
for grafica in ['wordcloud_frecuencias.png', 'top30_palabras_clave.png', 'tfidf_palabras_importantes.png']:
    if os.path.exists(grafica):
        try:
            import shutil
            shutil.copy(grafica, f'{RESULTS_PATH}/{grafica}')
            logger.info(f'[SAVE] ✅ {grafica}')
        except:
            pass

logger.info(f'\n[RESULTS] Guardado en: {RESULTS_PATH}/')


In [ ]:
# CELDA 10: Validación y estadísticas
logger.info(f'\n[VALIDACIÓN] FASE 3 COMPLETADO')
logger.info(f'\n✅ Eventos: {len(corpus_salida)}')
logger.info(f'✅ Artículos totales: {sum(len(e["articulos"]) for e in corpus_salida)}')
logger.info(f'✅ Medios: {set(sum([e["medios"] for e in corpus_salida], []))}')
logger.info(f'✅ Gráficas: 3 (Word Cloud, TOP 30, TF-IDF)')
logger.info(f'\n✓ Listo para FASE 4: Análisis de sesgos')
